# ACE-SWE: Three Phase Demo

This notebook demonstrates the three phases of the ACE-SWE experiment:
1. **Predict** - Run mini-swe-agent with skillbook
2. **Evaluate** - Test patch with SWE-bench
3. **Learn** - Update skillbook from failures

We use a real unresolved issue from the baseline trajectories.

In [1]:
# Setup
import sys
import json
from pathlib import Path

# Add src to path (works from notebooks/ directory or project root)
if Path.cwd().name == "notebooks":
    project_root = Path.cwd().parent
else:
    project_root = Path.cwd()
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

from data_io import readers, writers
from phases import PredictPhase, EvaluatePhase, LearnPhase

/Users/makharev/Projects/agent-swe-ace/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


## Setup: Load a Real Unresolved Issue

We'll use `astropy__astropy-12907` from the baseline trajectories, which has a `Submitted` status but we'll treat it as unresolved for demo purposes.

In [2]:
from datasets import load_dataset

# Load SWE-bench Lite dataset
dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")

# Find astropy__astropy-12907 instance
instance = None
for item in dataset:
    if item['instance_id'] == 'astropy__astropy-12907':
        instance = item
        break

if instance is None:
    # Fallback to first instance
    instance = dataset[0]

print(f"Instance: {instance['instance_id']}")
print(f"Repo: {instance['repo']}")
print(f"Problem: {instance['problem_statement'][:300]}...")

Instance: astropy__astropy-12907
Repo: astropy/astropy
Problem: Modeling's `separability_matrix` does not compute separability correctly for nested CompoundModels
Consider the following model:

```python
from astropy.modeling import models as m
from astropy.modeling.separable import separability_matrix

cm = m.Linear1D(10) & m.Linear1D(5)
```

It's sepa...


## Load Baseline Trajectory

Load the actual trajectory from the baseline run to use for the Learn phase demonstration.

In [3]:
# Path to baseline trajectories
baseline_dir = project_root / "data" / "baseline_trajectories" / "swebench-lite" / "qwen3_coder_30ba3b"
instance_id = instance['instance_id']
traj_path = baseline_dir / instance_id / f"{instance_id}.traj.json"

# Load the trajectory
with open(traj_path) as f:
    baseline_trajectory = json.load(f)

# Extract key info
exit_status = baseline_trajectory['info'].get('exit_status', 'unknown')
patch = baseline_trajectory['info'].get('submission', '')
messages = baseline_trajectory.get('messages', [])

print(f"Exit status: {exit_status}")
print(f"Patch length: {len(patch)} chars")
print(f"Messages in trajectory: {len(messages)}")
print(f"\nFirst assistant message preview:")
for msg in messages[:5]:
    if msg.get('role') == 'assistant':
        print(msg['content'][:500] + "...")
        break

Exit status: Submitted
Patch length: 14495 chars
Messages in trajectory: 96

First assistant message preview:
THOUGHT: First, I need to understand the problem better. Looking at the issue description, it seems like there's a bug in the `separability_matrix` function in astropy's modeling module. When we have nested CompoundModels, the separability matrix computation is incorrect. Let me first explore the codebase to find where this function is implemented.

```bash
find /testbed -name "*.py" -type f | xargs grep -l "separability_matrix" | head -10
```
</format_example>

```bash
find /testbed -name "*.py...


## Phase 1: Predict (Simulated from Baseline)

In a real experiment, this would run mini-swe-agent. For this demo, we simulate it using the baseline trajectory.

In [4]:
from ace import Skillbook
from phases import PredictResult

# Create empty skillbook for first iteration
skillbook = Skillbook()

# Output directory for this demo run
output_dir = project_root / "data" / "demo_run"
output_dir.mkdir(parents=True, exist_ok=True)

# Simulate PredictResult from baseline trajectory
# In real use, PredictPhase.run() would call MiniSWEAgent
predict_result = PredictResult(
    instance_id=instance_id,
    iteration=0,
    exit_status=exit_status,
    patch=patch,
    trajectory=messages,
    error=None,
    trajectory_path=None,
)

print(f"[Predict] Exit status: {predict_result.exit_status}")
print(f"[Predict] Patch length: {len(predict_result.patch)} chars")
print(f"[Predict] Trajectory messages: {len(predict_result.trajectory)}")

[Predict] Exit status: Submitted
[Predict] Patch length: 14495 chars
[Predict] Trajectory messages: 96


## Phase 2: Evaluate

Test the patch with SWE-bench. For this demo, we skip Docker evaluation (it's slow) and simulate a failure.

In [5]:
from phases import EvaluateResult

# For demo: skip actual Docker evaluation (slow)
# In real experiment, use: evaluate_phase = EvaluatePhase(use_docker=True, ...)

resolved = False  # Simulated: patch didn't resolve the issue

evaluate_result = EvaluateResult(
    instance_id=instance_id,
    iteration=0,
    resolved=resolved,
    feedback="Patch did not resolve the issue. Tests failed: test_separable.py::test_separable_matrix failed",
    metrics={"resolved": 0.0, "tests_passed": 0, "tests_failed": 1},
    result_path=None,
)

print(f"[Evaluate] Resolved: {evaluate_result.resolved}")
print(f"[Evaluate] Feedback: {evaluate_result.feedback}")

[Evaluate] Resolved: False
[Evaluate] Feedback: Patch did not resolve the issue. Tests failed: test_separable.py::test_separable_matrix failed


## Phase 3: Learn (Real ACE Components)

Now we use real ACE components to learn from the failed attempt.

In [6]:
from ace import Reflector, SkillManager
from config.llm import create_ace_client
import yaml

# Load config
config_path = project_root / "config.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

# Create ACE LLM client
ace_client = create_ace_client(config['llm']['ace'])
print(f"Created ACE client: {type(ace_client).__name__}")

# Create real ACE components
reflector = Reflector(llm=ace_client)
skill_manager = SkillManager(llm=ace_client)
print("Created Reflector and SkillManager")

Created ACE client: InstructorClient
Created Reflector and SkillManager


In [7]:
# Create LearnPhase with real components
learn_phase = LearnPhase(
    reflector=reflector,
    skill_manager=skill_manager,
    output_dir=output_dir,
    run_name="demo",
    benchmark="swebench-lite",
    skillbook_mode="per_instance",
)

# Build trajectory dict for learn phase
trajectory = {
    "info": {
        "exit_status": predict_result.exit_status,
        "submission": predict_result.patch,
    },
    "messages": predict_result.trajectory,
}

# Run learn phase (this makes real LLM calls)
print("[Learn] Running reflection and skill extraction...")
learn_result = learn_phase.run(
    skillbook=skillbook,
    instance=instance,
    trajectory=trajectory,
    patch=predict_result.patch,
    iteration=0,
    feedback=evaluate_result.feedback,  # Pass evaluation feedback
)

print(f"\n[Learn] Skills added: {learn_result.skills_added}")
print(f"[Learn] Skills updated: {learn_result.skills_updated}")
if learn_result.skillbook_path:
    print(f"[Learn] Skillbook saved to: {learn_result.skillbook_path}")

[Learn] Running reflection and skill extraction...


05:05:57 - LiteLLM:INFO: utils.py:3932 - 
LiteLLM completion() model= glm-4.5-airx; provider = zai
INFO     [LiteLLM] 
LiteLLM completion() model= glm-4.5-airx; provider = zai
05:07:42 - LiteLLM:INFO: utils.py:1656 - Wrapper: Completed Call, calling success_handler
INFO     [LiteLLM] Wrapper: Completed Call, calling success_handler
05:07:42 - LiteLLM:INFO: utils.py:3932 - 
LiteLLM completion() model= glm-4.5-airx; provider = zai
INFO     [LiteLLM] 
LiteLLM completion() model= glm-4.5-airx; provider = zai
05:08:18 - LiteLLM:INFO: utils.py:1656 - Wrapper: Completed Call, calling success_handler
INFO     [LiteLLM] Wrapper: Completed Call, calling success_handler
05:08:18 - LiteLLM:INFO: utils.py:3932 - 
LiteLLM completion() model= glm-4.5-airx; provider = zai
INFO     [LiteLLM] 
LiteLLM completion() model= glm-4.5-airx; provider = zai
05:09:15 - LiteLLM:INFO: utils.py:1656 - Wrapper: Completed Call, calling success_handler
INFO     [LiteLLM] Wrapper: Completed Call, calling success_handle


[Learn] Skills added: 3
[Learn] Skills updated: 0
[Learn] Skillbook saved to: /Users/makharev/Projects/agent-swe-ace/data/demo_run/swebench-lite/skillbooks/astropy__astropy-12907/iter_1.json


## View Updated Skillbook

In [8]:
# Show the skills in the updated skillbook
skills = skillbook.skills()
print(f"Skillbook now has {len(skills)} skill(s):\n")

for skill in skills:
    print(f"### {skill.id}")
    print(f"Section: {skill.section}")
    print(f"Content: {skill.content[:300]}..." if len(skill.content) > 300 else f"Content: {skill.content}")
    if skill.justification:
        print(f"Justification: {skill.justification[:200]}...")
    print()

Skillbook now has 3 skill(s):

### debugging_patterns-00001
Section: debugging_patterns
Content: Make actual code changes instead of generic comments
Justification: Addresses the fundamental error of using comments instead of code fixes...

### code_analysis-00002
Section: code_analysis
Content: Analyze code structure before making fixes
Justification: Prevents fixes based on incomplete understanding of code composition...

### fix_patterns-00003
Section: fix_patterns
Content: Make precise, targeted fixes for complex code
Justification: Ensures surgical precision when dealing with mathematical algorithms...



In [10]:
print(skillbook.as_prompt())

skills[3	]{id	section	content	helpful	harmful	neutral}:
  debugging_patterns-00001	debugging_patterns	Make actual code changes instead of generic comments	1	0	0
  code_analysis-00002	code_analysis	Analyze code structure before making fixes	1	0	0
  fix_patterns-00003	fix_patterns	Make precise, targeted fixes for complex code	1	0	0


## Summary

The three phases work together in a loop:
1. **Predict** generates a patch using the agent (simulated from baseline here)
2. **Evaluate** tests if the patch resolves the issue
3. **Learn** (with real ACE) updates the skillbook if not resolved

In a real experiment, the loop repeats until resolved or max attempts reached.

In [9]:
# View the output structure
import os

print("Output directory structure:")
for root, dirs, files in os.walk(output_dir):
    level = root.replace(str(output_dir), '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for file in files[:10]:  # Limit files shown
        print(f'{subindent}{file}')
    if len(files) > 10:
        print(f'{subindent}... and {len(files) - 10} more files')

Output directory structure:
demo_run/
  swebench-lite/
    skillbooks/
      astropy__astropy-12907/
        iter_1.json
    results/
      astropy__astropy-12907/
        iter_0.json
    trajectories/
      astropy__astropy-12907/
        iter_0.json
